# Tarea 73 - Airfoil Self-Noise

El dataset Airfoil Self-Noise es un conjunto de datos clásico de la NASA (donado al repositorio UCI) que proviene de pruebas aerodinámicas en túneles de viento. Su objetivo es modelar el ruido generado por el aire al pasar por el borde de un perfil alar.

ReLU

## Imports

In [67]:
import pandas as pd
import matplotlib.pyplot as plt
import os

import torch
from torch.utils.data import Dataset
from torch.utils.data import random_split
import torch.nn.functional as F
import torch.nn as nn

from torch.utils.tensorboard import SummaryWriter

In [68]:
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# device = torch.device('cpu')
print('Using device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

torch.backends.cuda.matmul.allow_tf32 = True

# The flag below controls whether to allow TF32 on cuDNN. This flag defaults to True.
torch.backends.cudnn.allow_tf32 = True

Using device: cpu


## Procesamiento de datos

In [69]:
airfoil = pd.read_csv("../data/airfoil_self_noise.dat", sep='\\s+', header = None)
airfoil

,0,1,2,3,4,5
0,800,0.0,0.3048,71.3,0.002663,126.201
1,1000,0.0,0.3048,71.3,0.002663,125.201
2,1250,0.0,0.3048,71.3,0.002663,125.951
3,1600,0.0,0.3048,71.3,0.002663,127.591
4,2000,0.0,0.3048,71.3,0.002663,127.461
...,...,...,...,...,...,...
1498,2500,15.6,0.1016,39.6,0.052849,110.264
1499,3150,15.6,0.1016,39.6,0.052849,109.254
1500,4000,15.6,0.1016,39.6,0.052849,106.604
1501,5000,15.6,0.1016,39.6,0.052849,106.224


### Escalar

In [70]:
class StandardScaler:

    def __init__(self, mean=None, std=None, epsilon=1e-7):
        """Standard Scaler.
        The class can be used to normalize PyTorch Tensors using native functions. The module does not expect the
        tensors to be of any specific shape; as long as the features are the last dimension in the tensor, the module
        will work fine.
        :param mean: The mean of the features. The property will be set after a call to fit.
        :param std: The standard deviation of the features. The property will be set after a call to fit.
        :param epsilon: Used to avoid a Division-By-Zero exception.
        """
        self.mean = mean
        self.std = std
        self.epsilon = epsilon
    def fit(self, values):
        dims = list(range(values.dim() - 1))
        self.mean = torch.mean(values, dim=dims)
        self.std = torch.std(values, dim=dims)
    def transform(self, values):
        return (values - self.mean) / (self.std + self.epsilon)

    def fit_transform(self, values):
        self.fit(values)
        return self.transform(values)

    def __repr__(self):
        return f"mean: {self.mean}, std:{self.std}, epsilon:{self.epsilon}"

### Dataset Airfoil

In [71]:
class AirfoilDataset(Dataset):
  def __init__(self, src_file, root_dir, transform=None):
    airfoilDataset = pd.read_csv(src_file, sep=r'\s+', names=['frequency', 'angle_of_attack', 'chord_length', 'velocity', 'displacement_thickness', 'sound_pressure_level'])
    X = airfoilDataset.loc[:, ~airfoilDataset.columns.isin(['sound_pressure_level'])]
    Y = airfoilDataset[["sound_pressure_level"]]
    
    # Convertir a tensores
    x_tensor = torch.tensor(X.values, dtype=torch.float32, device=device) # Cuda, añadimos device
    y_tensor = torch.tensor(Y.values, dtype=torch.float32, device=device)
    
    # TODO: aperez: Guardamos el sacaler para utilizarlo después en predicciones
    self.scaler = StandardScaler()
    self.scaler.fit(x_tensor)
    XScalada = self.scaler.fit_transform(x_tensor).type(torch.float32)
    self.data = torch.cat((XScalada,y_tensor),1)

    self.root_dir = root_dir
    self.transform = transform

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    if torch.is_tensor(idx):
      idx = idx.tolist()
    preds = self.data[idx, :5] # TODO: aperez: Property dataset - target (6-1 =5)
    spcs = self.data[idx, 5].float()
    sample = (preds, spcs)
    if self.transform:
      sample = self.transform(sample)
    return sample

### Cargar Airfoil

In [72]:
dataset = AirfoilDataset("../data/airfoil_self_noise.dat", root_dir=None)
display(dataset[0])
print(dataset.data.shape)

(tensor([-0.6618, -1.1460,  1.7987,  1.3125, -0.6446]), tensor(126.2010))

torch.Size([1503, 6])


## División en Train y Test

In [73]:
lonxitudeDataset = len(dataset)
tamTrain =int(lonxitudeDataset*0.8)
tamVal = lonxitudeDataset - tamTrain

train_set, val_set = random_split(dataset, [tamTrain, tamVal]) # TODO: aperez: Faltaba esta linea para futuros proyectos

print(f"Tam dataset: {lonxitudeDataset} train: {tamTrain} tamVal: {tamVal}")
train_ldr = torch.utils.data.DataLoader(train_set, batch_size=2,
    shuffle=True, drop_last=False)
# TODO: aperez: num_workers=2 -> S.O: Linux o macOS, PyTorch puede crear "clones" (forks) del proceso principal muy fácilmente para que carguen los datos en paralelo
#               En Windowns maneja el multiprocesamiento de forma diferente (usa algo llamado spawn). Al intentar crear estos subprocesos dentro de un Notebook, 
#               Windows se lía intentando recargar todo tu código desde cero, y los trabajadores colapsan. Ponemos a 0!!

# validation_loader =torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=False, num_workers=2)
validation_loader =torch.utils.data.DataLoader(val_set, batch_size=4, shuffle=False, num_workers=0)   

Tam dataset: 1503 train: 1202 tamVal: 301


## Creación de Modelo

In [74]:
class Model(nn.Module):
    def __init__(self, input_dim):
        super(Model, self).__init__()
        self.layer1 = nn.Linear(input_dim, 50)
        self.layer2 = nn.Linear(50, 50)
        self.layer3 = nn.Linear(in_features=50, out_features=1)
        
    def forward(self, x):
        x = F.relu(self.layer1(x))
        x = F.relu(self.layer2(x))
        x = self.layer3(x)
        return x

## Instanciar el Modelo

In [75]:
model = Model(5)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = torch.nn.MSELoss() 
display(model)

Model(
  (layer1): Linear(in_features=5, out_features=50, bias=True)
  (layer2): Linear(in_features=50, out_features=50, bias=True)
  (layer3): Linear(in_features=50, out_features=1, bias=True)
)

In [76]:
entradaProba,dest = next(iter(train_ldr))
print("Entrada:")
display(entradaProba)
print("Desexada:")
display(dest)
saida = model(entradaProba) # esta é a proba de verdade
print("Saída:")
display(saida)
loss_fn(saida, dest.view(-1, 1).float())

Entrada:


tensor([[-0.7887,  0.2733, -0.9167, -0.7231, -0.4165],
        [-0.6618,  1.4899, -0.3736,  1.3125,  2.4780]])

Desexada:


tensor([120.0150, 124.1880])

Saída:


tensor([[-0.0745],
        [-0.1685]], grad_fn=<AddmmBackward0>)

tensor(14943.0146, grad_fn=<MseLossBackward0>)

## Entrenamiento

In [77]:
def train_one_epoch(epoch_index, tb_writer):
    running_loss = 0.
    last_loss = 0.
    # usamos enumerate para saber en que batch imos
    for i, data in enumerate(train_ldr):
        # Every data instance is an input + label pair
        inputs, labels = data
        # Zero your gradients for every batch!
        optimizer.zero_grad()
        # Make predictions for this batch
        outputs = model(inputs)
        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels.view(-1, 1).float())
        loss.backward()
        # Adjust learning weights
        optimizer.step()
        # Gather data and report
        running_loss += loss.item()
        if i % 10 == 9:
            last_loss = running_loss / 10 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.
    return last_loss

In [78]:
writer = SummaryWriter('runs/experimento_airfoil')

EPOCHS = 100
loss_list = torch.zeros((EPOCHS,))
mae_list  = torch.zeros((EPOCHS,))

for epoch in range(EPOCHS):
    print(f'EPOCH {epoch + 1}:')

    model.train(True)
    avg_loss = train_one_epoch(epoch, writer)
    loss_list[epoch] = avg_loss

    # Esto habilitará la pestaña "Histograms" en TensorBoard
    for name, param in model.named_parameters():
        writer.add_histogram(name, param, epoch + 1)

    # Validación
    model.train(False)
    running_vloss = 0.0
    running_mae = 0.0
    
    with torch.no_grad():
        for i, vdata in enumerate(validation_loader):
            vinputs, vlabels = vdata
            voutputs = model(vinputs)
            
            # Ajuste de dimensiones y tipo para regresión
            vlabels_v = vlabels.view(-1, 1).float()
            
            vloss = loss_fn(voutputs, vlabels_v)
            running_vloss += vloss.item()

            # Calculamos el error real (MAE) para TensorBoard
            batch_mae = torch.abs(voutputs - vlabels_v).mean()
            running_mae += batch_mae.item()

    avg_vloss = running_vloss / (i + 1)
    avg_mae = running_mae / (i + 1)
    mae_list[epoch] = avg_mae

    # Enviar datos a Tensorboard
    # Esto creará las gráficas que verás en el navegador
    writer.add_scalars('Loss_Train_vs_Valid', 
                       {'Training': avg_loss, 'Validation': avg_vloss}, 
                       epoch + 1)
    writer.add_scalar('MAE_Validation', avg_mae, epoch + 1)
    
    print(f'LOSS train {avg_loss:.4f} valid {avg_vloss:.4f} | MAE: {avg_mae:.2f}')

# Forzar escritura y cerrar
# Sin esto, TensorBoard puede salir en blanco
writer.flush()
writer.close()
print("Entrenamiento finalizado y datos guardados en TensorBoard.")

EPOCH 1:
  batch 10 loss: 15312.9701171875
  batch 20 loss: 15681.7390625
  batch 30 loss: 15326.54892578125
  batch 40 loss: 15793.53466796875
  batch 50 loss: 15341.8052734375
  batch 60 loss: 15296.64150390625
  batch 70 loss: 15418.9556640625
  batch 80 loss: 14319.05283203125
  batch 90 loss: 14473.69482421875
  batch 100 loss: 13540.57763671875
  batch 110 loss: 14275.4048828125
  batch 120 loss: 12400.03583984375
  batch 130 loss: 12060.9037109375
  batch 140 loss: 11410.8326171875
  batch 150 loss: 10322.934326171875
  batch 160 loss: 9022.707373046875
  batch 170 loss: 9055.60439453125
  batch 180 loss: 7207.510888671875
  batch 190 loss: 6756.7955078125
  batch 200 loss: 5137.9390869140625
  batch 210 loss: 5159.5562744140625
  batch 220 loss: 3410.1447631835936
  batch 230 loss: 3376.14873046875
  batch 240 loss: 1375.2943023681642
  batch 250 loss: 1508.4644775390625
  batch 260 loss: 1136.82451171875
  batch 270 loss: 962.5293807983398
  batch 280 loss: 784.0152410507202
 

## Persistencia del Modelo (Exportación del state_dict)

In [79]:

# Definimos el nombre de la carpeta y creamos la ruta completa
carpeta_modelos = 'modelos'
ruta_archivo = os.path.join(carpeta_modelos, 'modelo_airfoil_entrenado.pth')

os.makedirs(carpeta_modelos, exist_ok=True)

# Guardamos el "cerebro" de la red en esa nueva ruta
torch.save(model.state_dict(), ruta_archivo)

print(f"¡Modelo guardado exitosamente en la ruta: {ruta_archivo}!")

¡Modelo guardado exitosamente en la ruta: modelos/modelo_airfoil_entrenado.pth!


## Predicción

In [80]:
model.load_state_dict(torch.load('modelos/modelo_airfoil_entrenado.pth'))
model.eval() # Modo evaluación (importante para desactivar Dropout/Batchnorm si los hubiera)

print("Modelo cargado correctamente.")

Modelo cargado correctamente.


In [ ]:
# 1Aseguramos que el modelo esté en modo evaluación
model.eval()

# Definimos los datos de entrada "brutos" (los reales)
# [frecuencia, angulo, cuerda, velocidad, espesor]
# Recogemos la primera fila de airfoil_self_noise.dat!!
input_raw = torch.tensor([[800.0, 0.0, 0.3048, 71.3, 0.00266337]], dtype=torch.float32, device=device)

# Usamos el scaler que vive dentro de tu dataset
# Esto garantiza que el "idioma" sea el mismo que el del entrenamiento
input_escalado = dataset.scaler.transform(input_raw)

# Realizamos la predicción
with torch.no_grad():
    prediccion = model(input_escalado)

# Mostramos los resultados
print(f"--- Resultado Final ---")
print(f"Predicción del modelo: {prediccion.item():.3f} dB")
print(f"Valor real esperado: 126.201 dB")
print(f"Error absoluto: {abs(prediccion.item() - 126.201):.3f} dB")

--- Resultado Final ---
Predicción del modelo: 123.325 dB
Valor real esperado: 126.201 dB
Error absoluto: 2.876 dB
